In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from xgboost import XGBClassifier

from sklearn.metrics import (precision_recall_curve, auc, classification_report, 
                             average_precision_score, confusion_matrix)

df = pd.read_csv(r"C:\Users\rchan\Downloads\feature_engineered.csv")

# Drop columns that cause Data Leakage
leakage_cols = ['DateTime', 'Time', 'Date', 'Sender_account', 'Receiver_account', 'Laundering_type']
df = df.drop(columns=[col for col in leakage_cols if col in df.columns], errors='ignore')

# Separate Features (X) and Target (y)
X = df.drop('Is_laundering', axis=1)
y = df['Is_laundering']


X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.15/0.85, stratify=y_temp, random_state=42)

# --- A. Logistic Regression Baseline ---
lr_model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr_model.fit(X_train_smote, y_train_smote)
y_val_probs_lr = lr_model.predict_proba(X_val_processed)[:, 1]

# --- B. Random Forest Classifier ---
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1, class_weight='balanced')
rf_model.fit(X_train_smote, y_train_smote)
y_val_probs_rf = rf_model.predict_proba(X_val_processed)[:, 1]

# --- C. XGBoost Classifier ---
xgb_model = XGBClassifier(n_estimators=150, max_depth=6, learning_rate=0.1, 
                          scale_pos_weight=(len(y_train)-sum(y_train))/sum(y_train), # Alternative to SMOTE for trees
                          random_state=42, eval_metric='aucpr', n_jobs=-1)
xgb_model.fit(X_train_processed, y_train) # Training without SMOTE as scale_pos_weight is used
y_val_probs_xgb = xgb_model.predict_proba(X_val_processed)[:, 1]

# --- D. Isolation Forest (Unsupervised) ---
# Train only on normal transactions (class 0) from training set
X_train_normal = X_train_processed[y_train == 0]
iso_model = IsolationForest(contamination=0.01, random_state=42, n_jobs=-1)
iso_model.fit(X_train_normal)
# Score: lower means more anomalous. We invert to align with predict_proba logic (higher = fraud)
y_val_scores_iso = -iso_model.score_samples(X_val_processed)

models_preds = {
    'Logistic Regression': y_val_probs_lr,
    'Random Forest': y_val_probs_rf,
    'XGBoost': y_val_probs_xgb,
    'Isolation Forest': y_val_scores_iso
}

for model_name, y_probs in models_preds.items():
    precision, recall, _ = precision_recall_curve(y_val, y_probs)
    pr_auc = auc(recall, precision)
    auprc_scores[model_name] = pr_auc



print("\nAUPRC Scores:")
for model, score in auprc_scores.items():
    print(f"{model}: {score:.4f}")

# ==========================================
# 9. Hyperparameter Optimization (Example on XGBoost)
# ==========================================
print("\nRunning Hyperparameter Optimization for XGBoost...")
param_grid = {
    'max_depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [100, 200],
    'subsample': [0.8, 1.0]
}

# Stratified K-Fold cross-validation
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
random_search = RandomizedSearchCV(XGBClassifier(scale_pos_weight=(len(y_train)-sum(y_train))/sum(y_train), random_state=42, eval_metric='aucpr'), 
                                   param_distributions=param_grid, n_iter=5, scoring='average_precision', 
                                   cv=cv, n_jobs=-1, random_state=42)

random_search.fit(X_train_processed, y_train)
best_xgb = random_search.best_estimator_
print(f"Best XGBoost Parameters: {random_search.best_params_}")




AUPRC Scores:
Logistic Regression: 0.3113
Random Forest: 0.2949
XGBoost: 0.6495
Isolation Forest: 0.0021

Running Hyperparameter Optimization for XGBoost...
Best XGBoost Parameters: {'subsample': 1.0, 'n_estimators': 200, 'max_depth': 8, 'learning_rate': 0.1}
